In [1]:
# 4_feature_eng_ukhls.ipynb
#
# Applies feature engineering to the backfilled UKHLS Wave O data.
# All operations are driven by ukhls_variables.py — no hardcoded lists.
#
# Steps:
#   1. Load o_indresp_backfilled.pkl
#   2. Apply value recodes (RECODE_MAPS)     e.g. hiqual_dv 9 → 5
#   3. Apply floor clipping  (FLOOR_VALUES)  e.g. payn_dv < 0 → 0
#   4. Apply upper clipping  (CLIP_VALUES)   e.g. carmiles > 50,000 → 50,000
#   5. Select cluster features + pidp (CLUSTER_VARS)
#   6. Force all columns to float32 (no categoricals)
#   7. Save as o_indresp_feature_eng.pkl
#
# Output feeds directly into 5_normalise_ukhls.ipynb

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import importlib
import data_processing.ukhls_variables as _ukhls_vars
importlib.reload(_ukhls_vars)

import pandas as pd
import numpy as np

from data_processing.ukhls_variables import (
    CLUSTER_VARS,
    RECODE_MAPS,
    FLOOR_VALUES,
    CLIP_VALUES,
)

# ── Config ────────────────────────────────────────────────────────────────────
WAVE       = "o"
INPUT_PKL  = "../data/3_backfill_ukhls_waves/o_indresp_backfilled.pkl"
OUTPUT_PKL = "../data/4_feature_eng_ukhls/o_indresp_feature_eng.pkl"

def get_base_code(col_name, wave_prefix):
    prefix = f"{wave_prefix}_"
    return col_name[len(prefix):] if col_name.startswith(prefix) else col_name

# ── 1. Load ───────────────────────────────────────────────────────────────────
print(f"Loading {INPUT_PKL} ...")
df = pd.read_pickle(INPUT_PKL)
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")

# ── 2. Value Recodes ──────────────────────────────────────────────────────────
print("\nStep 2: Applying value recodes ...")
recoded = []
for col in df.columns:
    base = get_base_code(col, WAVE)
    recode = RECODE_MAPS.get(base)
    if recode:
        before = df[col].value_counts(dropna=False).to_dict()
        df[col] = pd.to_numeric(df[col], errors='coerce').replace(recode)
        recoded.append(f"  {col}: {recode}")
if recoded:
    print("\n".join(recoded))
else:
    print("  (none)")

# ── 3. Floor Clipping ─────────────────────────────────────────────────────────
print("\nStep 3: Applying floor values ...")
for col in df.columns:
    base = get_base_code(col, WAVE)
    floor_val = FLOOR_VALUES.get(base)
    if floor_val is not None:
        numeric = pd.to_numeric(df[col], errors='coerce')
        n_floored = int((numeric < floor_val).sum())
        df[col] = numeric.clip(lower=floor_val)
        print(f"  {col}: floored at {floor_val}  ({n_floored:,} rows affected)")

# ── 4. Upper Clipping ─────────────────────────────────────────────────────────
print("\nStep 4: Applying clip values ...")
for col in df.columns:
    base = get_base_code(col, WAVE)
    clip_val = CLIP_VALUES.get(base)
    if clip_val is not None:
        numeric = pd.to_numeric(df[col], errors='coerce')
        n_clipped = int((numeric > clip_val).sum())
        df[col] = numeric.clip(upper=clip_val)
        print(f"  {col}: clipped at {clip_val:,}  ({n_clipped:,} rows affected)")

# ── 5. Select cluster features + pidp ─────────────────────────────────────────
print("\nStep 5: Selecting cluster features ...")
keep_cols = ['pidp'] + [f"{WAVE}_{base}" for base in CLUSTER_VARS]
missing_cols = [c for c in keep_cols if c not in df.columns]
if missing_cols:
    print(f"  WARNING — columns not found in data: {missing_cols}")
keep_cols = [c for c in keep_cols if c in df.columns]
df = df[keep_cols]
print(f"  Retained {len(keep_cols)} columns  ({len(keep_cols) - 1} features + pidp)")

# ── 6. Force all to float32 ───────────────────────────────────────────────────
print("\nStep 6: Converting all features to float32 ...")
for col in df.columns:
    if col == 'pidp':
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(np.int64)
    else:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(np.float32)

n_nulls = df.drop(columns=['pidp']).isna().sum().sum()
if n_nulls > 0:
    print(f"  WARNING — {n_nulls:,} NaN values remain after feature engineering")
    print(df.drop(columns=['pidp']).isna().sum()[lambda s: s > 0])
else:
    print("  No NaN values — clean feature matrix")

# ── 7. Save ───────────────────────────────────────────────────────────────────
df.to_pickle(OUTPUT_PKL, protocol=5)
print(f"\nDone. Feature matrix saved to {OUTPUT_PKL}")
print(f"Shape: {df.shape}")
print(df.dtypes)

Loading ../data/3_backfill_ukhls_waves/o_indresp_backfilled.pkl ...
Loaded 47,354 rows × 52 columns

Step 2: Applying value recodes ...
  o_hiqual_dv: {9.0: 5.0}

Step 3: Applying floor values ...
  o_payn_dv: floored at 0  (0 rows affected)

Step 4: Applying clip values ...
  o_jbttwt: clipped at 120  (61 rows affected)
  o_carmiles: clipped at 50,000  (67 rows affected)
  o_fimngrs_dv: clipped at 8,000  (1,040 rows affected)
  o_payn_dv: clipped at 8,000  (54 rows affected)
  o_nchild_dv: clipped at 7  (2 rows affected)

Step 5: Selecting cluster features ...
  WARNING — columns not found in data: ['o_locsera']
  Retained 27 columns  (26 features + pidp)

Step 6: Converting all features to float32 ...
  WARNING — 27,773 NaN values remain after feature engineering
o_hiqual_dv     2001
o_payn_dv      25772
dtype: int64

Done. Feature matrix saved to ../data/4_feature_eng_ukhls/o_indresp_feature_eng.pkl
Shape: (47354, 27)
pidp                            int64
o_age_dv                   